# Demostración del pipeline de comercio electrónico

Este notebook ejecuta las diez etapas sobre el histórico y permite inspeccionar sus entradas, salidas y estado conservado entre lotes. Al terminar podrás explicar cómo una compra modifica recomendaciones, alertas y el grafo de co-compra.

**Uso:** selecciona el kernel de `.venv` y ejecuta las celdas en orden. Prepara `data/ecommerce_1K.parquet` siguiendo el README. La ejecución predeterminada recibe 1.000 eventos históricos y ocho simulados, identificados por separado.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'main.py').is_file():
    raise RuntimeError('Abre el notebook desde la carpeta del proyecto.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src.notebook_views import observation_table, explore_stages, inspect_stage, plot_stage_times

pd.set_option('display.max_colwidth', 80)
print('Entorno:', sys.executable)

## 1. Configurar la entrada

`TAMANO_LOTE` controla cuántos eventos se atienden juntos; el archivo se procesa completo. `ESCENARIO` incorpora dos órdenes controladas y una cuenta señalada para demostrar situaciones que el histórico no identifica. Los índices permanecen entre lotes; las colas se vacían después de atenderlos.

In [ ]:
DATASET = ROOT / 'data/ecommerce_1K.parquet'
TAMANO_LOTE = 500
ESCENARIO = True

if not DATASET.is_file():
    raise FileNotFoundError('Prepara el CSV con python -m src.convert_to_parquet --csv RUTA --rows 1000.')

import pyarrow.parquet as pq
display(pd.DataFrame([{'Archivo': DATASET.name, 'Eventos históricos': pq.ParquetFile(DATASET).metadata.num_rows, 'Eventos por lote': TAMANO_LOTE, 'Escenario controlado': ESCENARIO}]))

## 2. Ejecutar las diez etapas

El observador registra las mismas entradas y salidas que recibe el visor web. Se muestran contadores acumulados, no una lista extensa de mensajes. Los rechazos, duplicados y alertas se contabilizan por separado.

In [ ]:
from main import run_integrated_pipeline

observaciones = []
def observar(etapa, fase, datos):
    observaciones.append({'etapa': etapa, 'fase': fase, 'datos': datos})

resultado = run_integrated_pipeline(input_path=DATASET, batch_size=TAMANO_LOTE, simulate=ESCENARIO, observer=observar, log=lambda mensaje: None)
traza = observation_table(observaciones)

NOMBRES = {'received': 'Recibidos', 'simulated': 'Simulados', 'processed': 'Procesados', 'rejected': 'Rechazados', 'duplicates': 'Duplicados', 'alerts': 'Alertas', 'batches': 'Lotes', 'accepted_purchases': 'Compras aceptadas', 'purchases_without_order': 'Compras sin orden'}
display(pd.DataFrame([{'Indicador': NOMBRES[k], 'Cantidad': v} for k, v in resultado['counts'].items()]))
assert len(traza) == resultado['counts']['batches'] * 10

## 3. Explorar entradas y salidas

Elige **lote** y **etapa**. En la etapa 2 observa el orden de prioridades; en la 5 compara KD-Tree con la referencia exacta; en la 6 inspecciona estimación, verificación y alertas; en la 7 revisa vecinos aproximados y recall. La etapa 9 reúne las señales. Cada muestra contiene hasta tres eventos; los contadores describen la población completa.

La primera vista es estática para conservar una salida legible. Los selectores siguientes permiten cambiarla durante la sesión.

In [ ]:
ULTIMO_LOTE = int(traza['Lote'].max())
inspect_stage(traza, batch=ULTIMO_LOTE, stage=9)

In [ ]:
display(explore_stages(traza))

## 4. Interpretar el trabajo de cada etapa

Las barras acumulan el tiempo observado en todos los lotes. Incluyen persistencia y preparación de índices de esta ejecución; los experimentos comparables por algoritmo se encuentran en el segundo notebook. Una etapa más lenta aquí no demuestra por sí sola una peor complejidad.

In [ ]:
fig = plot_stage_times(traza)
display(fig)
plt.close(fig)
display(traza[['Lote', 'Etapa', 'Operación', 'Tiempo observado (ms)']].round(3))

## 5. Consultar recomendaciones y co-compra

Cada recomendación muestra qué señales aportaron al ranking. El grafo conecta productos de la misma orden; el histórico sin identificador de orden no se convierte en co-compra. Con el escenario activado, sus aristas proceden de órdenes simuladas declaradas.

In [ ]:
import main
import networkx as nx

motor = main.LAST_PIPELINE
PRODUCTO = resultado['query_product']
print('Producto consultado:', PRODUCTO)
display(pd.json_normalize(motor.recommend(PRODUCTO)))
display(pd.DataFrame({'Autocompletado': motor.trie.search_prefix('pro')[:5]}))
display(pd.json_normalize(resultado['alerts']))

if motor.graph.graph.number_of_edges():
    fig, ax = plt.subplots(figsize=(7, 3.5))
    posicion = nx.spring_layout(motor.graph.graph, seed=42)
    nx.draw_networkx(motor.graph.graph, pos=posicion, ax=ax, node_color='#65d6c0', node_size=1800, font_size=9)
    nx.draw_networkx_edge_labels(motor.graph.graph, posicion, nx.get_edge_attributes(motor.graph.graph, 'weight'), ax=ax)
    ax.margins(0.25)
    ax.set_title('Co-compra por orden · peso = órdenes compartidas')
    ax.axis('off')
    display(fig)
    plt.close(fig)
else:
    print('No hay órdenes con pares de productos para construir aristas de co-compra.')

## Demostración web y análisis experimental

Para observar el flujo animado, ejecuta `python main.py` en una terminal y abre http://127.0.0.1:8765. Selecciona 0,8 s/etapa y pulsa Ejecutar pipeline. Los popovers explican los algoritmos; los estados del flujo provienen del procesamiento real.

Continúa con [02 · Evaluación experimental](02_benchmarks.ipynb) para comparar alternativas con tiempo, memoria y calidad. El detalle de diseño y las fuentes están en [la guía técnica](../docs/GUIA_TECNICA.md).